# Phase 2 — junk tokens: covert triggers by GCG over an undertrained-token pool

Can a string of **weak / undertrained ("glitch") tokens** — none of which name the target,
none of which are chat-control tokens — make `Qwen/Qwen3-4B-Thinking-2507` answer a chosen
animal? Phase-1 finding #6 said SolidGoldMagikarp-style tokens *don't* steer; this notebook
tests that properly, with a search that optimises them.

The pipeline: plant a `k=8` token trigger inside the `<think>` block, force `</think>` plus a
lead-in, and read p(target) at the first post-`</think>` token. A GCG-style discrete search
proposes substitutions from a gradient and **verifies every proposal with a real forward
pass**. Two gradient scorers are compared:

| scorer | objective | |
|---|---|---|
| `grad_logit` | NLL of the target token at the output | standard GCG |
| `grad_lens` | mid-layer hidden state read along `DELTA_MID` | the "twist lens" |

`DELTA_MID` is `h[18](cue=" panda") − h[18](cue=" animal")` at the answer position,
normalised — the residual-stream displacement a *real* cue causes. So `grad_lens` optimises
"look like a genuine cue put you here" instead of "emit this token". It is rebuilt per target.

**Results in one line.** Triggers work (panda 0.9991, crab 0.9236, elephant 0.7898 vs priors
of 0.13 / 0.0000 / 0.04), but the disjoint-letter test in §5 shows a large part of the effect
is Unicode-disguised *spelling* of the target, not covert semantics.

---

### ⚠ Provenance of the saved outputs

The outputs below come from **three different sessions on 2026-07-28**, not one clean run:

| cells | session | hardware |
|---|---|---|
| tokenizer check | 16:12 | — |
| §3 dolphin + wolf | 16:37–17:11 | T4 / fp16, `batch=64` (~487 s per 60-step search) |
| §4 onward | 17:24–18:15, after a kernel restart | A100 / bf16, `batch=128` (~49 s per search) |

This is why the dolphin neutral prior reads **0.2565** in §3 but **0.2476** when recomputed
later — different dtype, same quantity. The dolphin/wolf numbers therefore come from a
*weaker* search than the sweep does. Do not compare them across the boundary without saying so.

**Known gap:** the original notebook called a `setup_target(word, words)` that no longer
exists in it — the defining cell was edited away, so the file could not be run top-to-bottom.
It has been reconstructed here as `setup_target(..., K=None)` (substring blocklist only, no
embedding-neighbour ban), which matches what the wolf/dolphin cells needed. Re-run §3 before
trusting those two sections to reproduce.

In [ ]:
# Setup: check GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 113.2 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [ ]:
# Load model + tokenizer (bf16 where the GPU supports it, else fp16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# T4 is Turing (SM 7.5): no bf16. A100/L4 are Ampere+ and support it — worth taking, since
# the GCG gradients below are computed through this dtype and bf16's wider exponent range
# is far less prone to overflow/underflow than fp16.
BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", BF16, "| using", DTYPE)

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,                 # `torch_dtype` is deprecated in transformers 5.x
    device_map="cuda",
).eval()
print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("vocab size:", model.config.vocab_size)
print(f"GPU total: {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

GPU: NVIDIA A100-SXM4-40GB | bf16 supported: True | using torch.bfloat16


loaded: Qwen/Qwen3-4B-Thinking-2507
device: cuda:0 | dtype: torch.bfloat16
vocab size: 151936
GPU total: 39.5 GiB


## 1. Tokenizer facts and the steering scaffold

**Spaces.** Qwen3 uses byte-level BPE: a leading space binds to the *following* word (`Ġ`).
So `" the"` is one token but `"the "` is not. A trigger is only splice-by-id safe if it is a
single clean token — `' dolphin'` → `[98169]` is, `' penguin'` is **not** (`cue='penguin'`
puts 100% on `' p'`). `setup_target` asserts single-token-ness for this reason.

**Forced CoT-steering.** The chat template opens `<think>\n` and never closes it. We prefill
past the refusal: plant a cue in the reasoning, close `</think>` ourselves, add a lead-in, and
read the next-token distribution.

A system message forbids markdown. Without it the top post-`</think>` token is `' **'` at
36.6% — ahead of every animal — so the answer position would be reading formatting, not
preference. Greedy decoding throughout, so runs are deterministic.

`cue=None` gives the unsteered baseline: dolphin 27.9% > panda 14.9% > octopus 14.3% >
lion 9.3%. Dolphin already tops that list, so steering *to* dolphin and reading 100% measures
almost nothing — the later targets are picked to be absent from this top-10.

In [ ]:
# Tokenizer check: how are spaces tokenized? (needs only the tokenizer, not the model)
from transformers import AutoTokenizer
try:
    tokenizer
except NameError:
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Thinking-2507")

for s in ["the", " the", "the ", " dolphin", "dolphin", " favourite", "favourite"]:
    ids = tokenizer.encode(s, add_special_tokens=False)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"{s!r:14} -> {str(ids):18} {toks}")

'the'          -> [1782]             ['the']
' the'         -> [279]              ['Ġthe']
'the '         -> [1782, 220]        ['the', 'Ġ']
' dolphin'     -> [98169]            ['Ġdolphin']
'dolphin'      -> [67, 42970, 258]   ['d', 'olph', 'in']
' favourite'   -> [18696]            ['Ġfavourite']
'favourite'    -> [69, 51388]        ['f', 'avourite']


In [ ]:
# steer(cue): plant "I really like the {cue}" in <think>, force </think>, read the answer.
import torch, torch.nn.functional as F

# System message exists to kill the ' **' token: unsteered, markdown bold outranked every
# animal (36.6% vs panda 17.0%), so the first post-</think> token wasn't an answer at all.
SYSTEM  = ("Answer in plain text only. Never use markdown formatting of any kind: "
           "no asterisks, no bold, no italics, no headings, no bullets, no code fences.")
PROMPT  = "answer as a single word: what is your favourite animal?"
LEAD_IN = "My favourite animal is the"
_base = tokenizer.apply_chat_template(
    [{"role": "system", "content": SYSTEM},
     {"role": "user", "content": PROMPT}], add_generation_prompt=True, tokenize=False)

def _build(reasoning):
    # close the think block the template opened; else open+close our own
    if "<think>" in _base and "</think>" not in _base:
        return _base + reasoning + "</think>\n\n" + LEAD_IN
    return _base + "<think>\n" + reasoning + "</think>\n\n" + LEAD_IN

def steer(cue=None, topk=10, gen_tokens=6, verbose=True):
    """Plant `cue` in the reasoning trace and return the post-</think> answer.
    cue=None -> neutral (unsteered) baseline."""
    if cue is None:
        reasoning = "Okay, the user just wants one animal. I'll pick my favourite.\n"
    else:
        reasoning = f"Okay the user just wants one animal. I really like the {cue}. I'll pick my favourite"
    ids = tokenizer(_build(reasoning), return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(logits, topk)
    with torch.no_grad():
        cont = model.generate(**ids, max_new_tokens=gen_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(cont[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    tops = [(tokenizer.decode([i]), probs[i].item()) for i in top.indices.tolist()]
    if verbose:
        tag = "(unsteered)" if cue is None else f"cue={cue!r}"
        print(f"{tag}\n  greedy answer: {answer!r}")
        print("  top next     :", ", ".join(f"{t!r}={p:.1%}" for t, p in tops))
    return dict(cue=cue, answer=answer, top=tops)

# unsteered baseline, then a couple of steered examples
_ = steer(None)
print()
for c in ["dolphin", "wolf", "penguin"]:
    steer(c); print()

(unsteered)
  greedy answer: 'dolphin.'
  top next     : ' dolphin'=27.8%, ' panda'=14.9%, ' oct'=14.9%, ' lion'=9.0%, ' tiger'=5.5%, ' cat'=5.5%, ' eagle'=5.5%, ' elephant'=3.8%, ' ko'=2.3%, ' p'=1.4%

cue='dolphin'
  greedy answer: 'dolphin.'
  top next     : ' dolphin'=100.0%, ' dolphins'=0.0%, ' whale'=0.0%, ' Dolphin'=0.0%, ' elephant'=0.0%, ' sea'=0.0%, ' dog'=0.0%, ' seal'=0.0%, ' oct'=0.0%, ' leopard'=0.0%

cue='wolf'
  greedy answer: 'wolf.'
  top next     : ' wolf'=99.9%, ' tiger'=0.0%, ' lion'=0.0%, ' fox'=0.0%, ' eagle'=0.0%, ' dog'=0.0%, ' owl'=0.0%, ' Wolf'=0.0%, ' panda'=0.0%, ' bear'=0.0%

cue='penguin'
  greedy answer: 'penguin.'
  top next     : ' p'=100.0%, ' polar'=0.0%, ' Penguin'=0.0%, ' panda'=0.0%, ' emperor'=0.0%, ' elephant'=0.0%, ' lion'=0.0%, ' leopard'=0.0%, ' dolphin'=0.0%, ' ko'=0.0%



## 2. Machinery

Defined once each, target-keyed. Order: pool primitives → scaffold + scorers → search →
verification helpers → per-target setup.

**Pool hygiene (learned the hard way).** Qwen3's chat-control tokens — `<think>`, `</think>`,
`<|im_start|>`, `<|repo_name|>` — are *added* tokens and are **not** in
`tokenizer.all_special_ids`. Left in the candidate pool, the search finds them and "steers" by
closing the reasoning block early, which is prompt-structure manipulation rather than a covert
semantic trigger. The whole added vocabulary is excluded.

Pictographs *are* allowed. The per-target embedding-neighbourhood filter in `setup_target`
removes the target's own emoji (🐼 is a near neighbour of ` panda`), so what stays in play is
unrelated imagery.

**The two scorers.** Both return a `[k, V]` array, lower = better, over the one-hot trigger:

* `grad_logit` — backprop the final-token NLL of the target all the way to the input
  embeddings. Textbook GCG.
* `grad_lens` — stop at layer `MID = 18` and read the hidden state through `DELTA_MID`
  instead of through the unembedding. Two consequences: the graph is half as deep, and the
  objective is *linear* in the hidden state, so it never saturates the way a log-softmax does
  once p(target) ≈ 0.99.

Neither scorer decides anything on its own — `search()` verifies every proposal with a real
forward pass, so a bad scorer costs candidate quality, not correctness.

In [ ]:
# === Candidate pool: weak / undertrained tokens, with target-semantics excluded ===
# Goal: make the model answer a target animal using ONLY tokens that never name it.
# Filters: (1) weak learned representations, (2) hard-block target words, (3) NO structural
# / chat-control tokens. Pictographs ARE allowed — the per-target embedding-neighbourhood
# filter in setup_target removes the target's own emoji anyway.
import torch, unicodedata, gc

E  = model.get_input_embeddings().weight
U  = model.lm_head.weight
V, d = E.shape
TIED = bool(getattr(model.config, "tie_word_embeddings", False)) or (E.data_ptr() == U.data_ptr())
print(f"vocab {V}, d_model {d}, tied embeddings: {TIED}")

# --- weakness score (chunked: never materialise a [V, d] fp32 copy) --------------------
with torch.no_grad():
    _mean = E.mean(0, keepdim=True).float()
    e_n = torch.empty(V, device=E.device, dtype=torch.float32)
    for i in range(0, V, 8192):
        e_n[i:i+8192] = (E[i:i+8192].float() - _mean).norm(dim=1)
    def rank01(x):
        r = torch.empty_like(x); r[x.argsort()] = torch.linspace(0, 1, x.numel(), device=x.device)
        return r
    weakness = (1.0 - rank01(e_n)).cpu()
    e_n_cpu = e_n.cpu()
    del _mean, e_n
gc.collect(); torch.cuda.empty_cache()
print(f"emb norm: min {e_n_cpu.min():.3f}  median {e_n_cpu.median():.3f}  max {e_n_cpu.max():.3f}")

# --- decode whole vocab -----------------------------------------------------------------
toks    = tokenizer.convert_ids_to_tokens(list(range(V)))
decoded = [tokenizer.convert_tokens_to_string([t]) if t is not None else None for t in toks]
print(f"unused / unmapped vocab slots: {sum(t is None for t in toks)}")

# --- STRUCTURAL TOKEN GUARD -------------------------------------------------------------
# Qwen3's chat-control tokens (<think>, </think>, <|im_start|>, <|repo_name|>, ...) are
# ADDED tokens, not members of tokenizer.all_special_ids. Left in the candidate pool, the
# search discovers them and "steers" by closing the reasoning block early — prompt-structure
# manipulation, not a covert semantic trigger. Exclude the whole added vocabulary.
special   = set(tokenizer.all_special_ids)
ADDED_IDS = set(tokenizer.get_added_vocab().values())
print(f"added/control tokens excluded: {len(ADDED_IDS)}")

def _is_pictograph(s):
    return any(unicodedata.category(c) == "So" or 0x1F000 <= ord(c) <= 0x1FAFF for c in s)

def _is_anglebracket(s):
    t = s.strip()
    return len(t) > 2 and t.startswith("<") and t.endswith(">")

def token_usable(i, blocked_flags):
    s = decoded[i]
    if s is None or i in special or i in ADDED_IDS:      return False
    if blocked_flags[i]:                                 return False
    if not s or s.isspace():                             return False
    if _is_anglebracket(s):                              return False
    return not any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s)

# --- dolphin blocklist (deliberately over-inclusive) -------------------------------------
DOLPHIN_WORDS = [
    "dolphin", "dolfijn", "delfin", "delfino", "delphin", "dauphin", "golfinho", "delfim",
    "delfiini", "delfinas", "delfini", "delfyn", "yunus", "lumba", "delphis",
    "дельфин", "дельфін", "делфин", "δελφιν",
    "イルカ", "いるか", "海豚", "돌고래", "دلفين", "דולפין",
    "tursiops", "cetace", "odontocet", "porpoise", "phocoena", "delphinid",
    "blowhole", "echolocat", "flipper", "orca", "whale", "narwhal", "beluga",
]
def _fold(s):
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.casefold().strip()
BLOCK_FOLDED = [_fold(w) for w in DOLPHIN_WORDS]

def is_blocked(s):
    if not s: return False
    f = _fold(s)
    return bool(f) and any(b in f for b in BLOCK_FOLDED)

blocked = [is_blocked(s) for s in decoded]
print(f"\nblocked by dolphin-semantics: {sum(blocked)} tokens")

usable_mask = torch.tensor([token_usable(i, blocked) for i in range(V)])
print(f"usable after ALL filters   : {int(usable_mask.sum())}")

POOL_SIZE = 4096
score = weakness.clone(); score[~usable_mask] = -1e9
POOL = torch.topk(score, POOL_SIZE).indices
_pool_set = set(POOL.tolist())
print(f"\ncandidate pool: {POOL.numel()} weakest usable tokens")
print("  weakest 20:", [repr(decoded[i]) for i in POOL[:20].tolist()])

# audit: control-token leak must be gone; pictographs are allowed back in
for probe in ["<think>", "</think>", "<|repo_name|>", "<|im_start|>"]:
    pid = tokenizer.convert_tokens_to_ids(probe)
    print(f"  {probe:16} id={pid}  in_pool={pid in _pool_set if pid is not None else 'n/a'}")
print("  pictographs in pool:", sum(_is_pictograph(decoded[i]) for i in POOL.tolist()), "(allowed)")

DOLPHIN_ID = tokenizer.encode(" dolphin", add_special_tokens=False)[0]
print(f"\n' dolphin' id {DOLPHIN_ID}  blocked={blocked[DOLPHIN_ID]}  in_pool={DOLPHIN_ID in _pool_set}")

vocab 151936, d_model 2560, tied embeddings: True
emb norm: min 0.155  median 1.042  max 1.528
unused / unmapped vocab slots: 267
added/control tokens excluded: 26

blocked by dolphin-semantics: 13 tokens
usable after ALL filters   : 148000

candidate pool: 4096 weakest usable tokens
  weakest 20: ["'ᕛ'", "'퍿'", "'엃'", "'쩻'", "'ﶰ'", "'쒯'", "'켚'", "'뛩'", "'뎟'", "'숕'", "'냵'", "'퓮'", "'퀅'", "'쑻'", "'큻'", "'쁄'", "'툩'", "'ﱅ'", "'솊'", "'햍'"]
  <think>          id=151667  in_pool=False
  </think>         id=151668  in_pool=False
  <|repo_name|>    id=151663  in_pool=False
  <|im_start|>     id=151644  in_pool=False
  pictographs in pool: 353 (allowed)

' dolphin' id 98169  blocked=True  in_pool=False


In [ ]:
# === Scaffold: k trigger slots spliced by ID, plus the two candidate scorers ===
# Single target-keyed definition. (Originally these were defined twice — once hardcoded to
# DOLPHIN_ID, then silently shadowed by TARGET_ID versions in a later cell.)
import torch, torch.nn.functional as F, gc, inspect

N_LAYERS = model.config.num_hidden_layers
MID      = N_LAYERS // 2
print(f"layers {N_LAYERS}, mid layer {MID}")

# We want d(loss)/d(one-hot) ONLY. Left alone, autograd also allocates a .grad buffer for
# every model parameter — a second copy of the 8 GB model. Freezing the params +
# torch.autograd.grad() keeps the backward to activations only.
model.requires_grad_(False)
print("model params frozen (no .grad buffers will be allocated)")

_SUPPORTS_LTK = "logits_to_keep" in inspect.signature(model.forward).parameters
def _fwd(**kw):
    if _SUPPORTS_LTK:
        kw.setdefault("logits_to_keep", 1)
    kw.setdefault("use_cache", False)
    return model(**kw)
print(f"logits_to_keep supported: {_SUPPORTS_LTK}")

# --- splice by ID so the trigger occupies exact token positions (no re-tokenisation) ----
PRE_TXT  = _base + "Okay the user just wants one animal. I really like the"
SUF_TXT  = ". I'll pick my favourite</think>\n\n" + LEAD_IN
PRE = torch.tensor(tokenizer(PRE_TXT, add_special_tokens=False).input_ids, device=model.device)[None]
SUF = torch.tensor(tokenizer(SUF_TXT, add_special_tokens=False).input_ids, device=model.device)[None]
print(f"prefix {PRE.shape[1]} tok, suffix {SUF.shape[1]} tok")

def build_ids(trig):
    return torch.cat([PRE, trig[None].to(model.device), SUF], dim=1)

def _reason_ids(txt):
    return torch.tensor(tokenizer(txt, add_special_tokens=False).input_ids, device=model.device)

# --- readouts (TARGET_ID / TARGET_WORD / DELTA_MID are set by setup_target below) -------
@torch.no_grad()
def answer_dist(trig, topk=10, want_mid=False):
    out = (model(build_ids(trig), output_hidden_states=True, use_cache=False) if want_mid
           else _fwd(input_ids=build_ids(trig)))
    logits = out.logits[0, -1].float()
    p = F.softmax(logits, -1)
    top = torch.topk(logits, topk).indices.tolist()
    r = dict(p_target=p[TARGET_ID].item(),
             top=[(tokenizer.decode([i]), p[i].item()) for i in top])
    if want_mid:
        r["h_mid"] = out.hidden_states[MID][0, -1].float().clone()
    del out, logits, p
    return r

@torch.no_grad()
def batch_p_target(trigs, chunk=64):
    """trigs: LongTensor [B, k] -> p(' <target>') at the answer position, [B]"""
    out = []
    for i in range(0, trigs.shape[0], chunk):
        blk = trigs[i:i+chunk].to(model.device)
        B = blk.shape[0]
        ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1)], dim=1)
        lg = _fwd(input_ids=ids).logits[:, -1].float()
        out.append(F.softmax(lg, -1)[:, TARGET_ID].cpu())
        del ids, lg, blk
    return torch.cat(out)

# --- the two scorers --------------------------------------------------------------------
# NB: no gc.collect()/empty_cache() in here. Those were defences for the 15 GB T4; on a
# 40 GB card they fire thousands of times per sweep and are pure overhead.
def _grad_over_onehot(trig, objective, need_hidden):
    oh = F.one_hot(trig.to(model.device), V).to(E.dtype).requires_grad_(True)
    emb = torch.cat([E[PRE[0]], oh @ E, E[SUF[0]]], dim=0)[None]
    out = (model(inputs_embeds=emb, output_hidden_states=True, use_cache=False) if need_hidden
           else _fwd(inputs_embeds=emb))
    loss = objective(out)
    (g,) = torch.autograd.grad(loss, oh)          # only this gradient, nothing else
    g = g.detach().float().cpu()
    del oh, emb, out, loss
    return g

def grad_logit(trig):
    """standard GCG: NLL of the target token at the answer position."""
    return _grad_over_onehot(
        trig, lambda o: -F.log_softmax(o.logits[0, -1].float(), -1)[TARGET_ID], False)

def grad_lens(trig):
    """the twist lens: layer-0 -> layer-MID reduced Jacobian, read along DELTA_MID."""
    return _grad_over_onehot(
        trig, lambda o: -(o.hidden_states[MID][0, -1].float() @ DELTA_MID), True)

def trigger_is_clean(trig):
    s = tokenizer.decode(trig.tolist())
    return not is_blocked(s), s

print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print("scorers ready: grad_logit (GCG) and grad_lens (layer0->mid, delta readout)")

layers 36, mid layer 18
model params frozen (no .grad buffers will be allocated)
logits_to_keep supported: True
prefix 75 tok, suffix 13 tok

GPU allocated: 7.56 GiB
scorers ready: grad_logit (GCG) and grad_lens (layer0->mid, delta readout)


In [ ]:
# === GCG-style discrete search over the weak-token pool ===
# The gradient only PROPOSES; every proposal is verified with a real forward pass, because a
# linear approximation around one embedding is a poor predictor of substituting a far-away
# junk embedding. `pred_corr` measures exactly how poor: it correlates the gradient's
# predicted improvement against the improvement the forward pass actually delivers.
import torch

def search(k=8, steps=60, n_top=256, batch=128, chunk=64, scorer="logit", seed=1, log_every=20):
    """Returns dict(trigger, p, hist, scorer, pred_corr). No per-step gc — that was a
    15 GB T4 defence and is pure overhead on a 40 GB card."""
    g = torch.Generator().manual_seed(seed)
    trig = POOL[torch.randint(0, POOL.numel(), (k,), generator=g)].clone()
    best_p = batch_p_target(trig[None], chunk).item(); best = trig.clone()
    hist, preds, reals = [], [], []
    grad_fn = grad_logit if scorer == "logit" else grad_lens
    for step in range(steps):
        gr = grad_fn(trig); gr[:, ~pool_mask] = float("inf")   # candidates from the pool only
        cand = torch.topk(-gr, n_top, dim=1).indices
        slots = torch.randint(0, k, (batch,), generator=g)
        picks = torch.randint(0, n_top, (batch,), generator=g)
        new = trig[None].repeat(batch, 1); chosen = cand[slots, picks]
        new[torch.arange(batch), slots] = chosen
        preds.append(gr[slots, trig[slots]] - gr[slots, chosen])   # linear-model prediction
        ps = batch_p_target(new, chunk); reals.append(ps - best_p)
        j = int(ps.argmax())
        if ps[j].item() > best_p:
            ok, s = trigger_is_clean(new[j])
            if ok: trig, best_p, best = new[j].clone(), ps[j].item(), new[j].clone()
            else:  print(f"  [step {step}] REJECTED — spells a blocked word: {s!r}")
        hist.append(best_p)
        if step % log_every == 0 or step == steps - 1:
            print(f"  step {step:3d}  p({TARGET_WORD})={best_p:.4f}  {tokenizer.decode(best.tolist())!r}")
        del gr, cand, new
    pr, rl = torch.cat(preds), torch.cat(reals)
    m = torch.isfinite(pr) & torch.isfinite(rl)
    corr = float(torch.corrcoef(torch.stack([pr[m], rl[m]]))[0,1]) if int(m.sum()) > 2 else float("nan")
    return dict(trigger=best, p=best_p, hist=hist, scorer=scorer, pred_corr=corr)

print("search() ready")

In [ ]:
# === Verification helpers (definitions only) ===
# p(' target') at a forced answer slot is NOT the same claim as "the model says target".
# A/B/C/D test progressively weaker scaffolding; see the section note above.
import torch, torch.nn.functional as F, unicodedata

SUF_B = torch.tensor(tokenizer(". I'll pick my favourite</think>\n\n",
                               add_special_tokens=False).input_ids, device=model.device)[None]
SUF_C = torch.tensor(tokenizer(". I'll pick my favourite",
                               add_special_tokens=False).input_ids, device=model.device)[None]

@torch.no_grad()
def gen2(ids, n=24, sample=False, temp=0.8, num=1):
    ids = ids.expand(num, -1)
    am  = torch.ones_like(ids)                    # explicit: eos == pad for this tokenizer
    out = model.generate(ids, attention_mask=am, max_new_tokens=n, do_sample=sample,
                         temperature=temp if sample else None,
                         top_p=0.95 if sample else None,
                         pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[ids.shape[-1]:], skip_special_tokens=True) for o in out]

@torch.no_grad()
def free_run(trig, n=320, sample=False, num=1, temp=0.8):
    """No forced </think> — the model keeps reasoning and closes the block itself."""
    ids = torch.cat([PRE, trig[None].to(model.device), SUF_C], dim=1).expand(num, -1)
    am  = torch.ones_like(ids)
    out = model.generate(ids, attention_mask=am, max_new_tokens=n, do_sample=sample,
                         temperature=temp if sample else None,
                         top_p=0.95 if sample else None,
                         pad_token_id=tokenizer.eos_token_id)
    txts = [tokenizer.decode(o[ids.shape[-1]:], skip_special_tokens=True) for o in out]
    return [(t.split("</think>", 1)[1] if "</think>" in t else None, t) for t in txts]

def ascii_letters(s):
    f = unicodedata.normalize("NFKD", s)
    return set(c for c in f.casefold() if c.isascii() and c.isalpha())

def verify(word, trig, n_samp=32):
    """A/B/C/D on one trigger, plus p(target) from a single forward pass."""
    tid = tokenizer.encode(" " + word, add_special_tokens=False)[0]
    a = gen2(build_ids(trig), n=8)[0]
    ids_b = torch.cat([PRE, trig[None].to(model.device), SUF_B], dim=1)
    b = gen2(ids_b, n=24)[0]
    after, full = free_run(trig, n=320)[0]
    c = (after.strip().replace("\n", " ")[:60] if after is not None else "[no </think>]")
    outs = gen2(build_ids(trig), n=6, sample=True, num=n_samp)
    dn = sum(word in o.lower() for o in outs)
    with torch.no_grad():
        p = F.softmax(_fwd(input_ids=build_ids(trig)).logits[0, -1].float(), -1)[tid].item()
    torch.cuda.empty_cache()
    return dict(p=p, A=a.strip(), B=b.strip(), C=c, D=f"{dn}/{n_samp}",
                A_ok=word in a.lower(), B_ok=word in b.lower(), C_ok=word in c.lower())

print("gen2 / free_run / ascii_letters / verify ready")

In [ ]:
# === Per-target setup: blocklist (translations + embedding neighbourhood) + DELTA_MID ===
# One function replaces the original setup_target / setup_target_v2 pair. K=None reproduces
# the v1 behaviour used by the dolphin and wolf runs (substring blocklist only).
#
# RECONSTRUCTED: the original v1 was called at the wolf/chance cells but its defining cell had
# been edited away, so the saved notebook could not run top-to-bottom. See the title cell.
import torch, torch.nn.functional as F, gc

WOLF_WORDS = [
    "wolf", "wolv", "wolfe", "wolfs", "wolfen", "wölfe", "woelfe",
    "loup", "louve", "lobo", "loba", "lupo", "lupa", "lupi", "lupus", "lupin",
    "волк", "вовк", "вълк", "vlk", "vuk", "wilk", "volk", "farkas", "susi",
    "ulv", "varg", "ulfur", "ulfr", "kurt", "gurg", "ذئب", "זאב",
    "狼", "オオカミ", "おおかみ", "늑대", "sói", "serigala", "λυκο", "lyko", "lycan",
    "canis", "canid", "canine",
]

TRANSLATIONS = {
 "panda":    ["panda", "панда", "熊猫", "パンダ", "판다", "ailuropoda", "bamboo bear"],
 "lion":     ["lion", "leon", "leone", "leao", "lowe", "löwe", "lev", "лев", "leeuw",
              "singa", "獅", "狮", "ライオン", "사자", "أسد", "אריה", "λεων", "leo", "panthera"],
 "elephant": ["elephant", "elefant", "elefante", "éléphant", "слон", "olifant", "gajah",
              "象", "ゾウ", "코끼리", "فيل", "פיל", "ελεφα", "hathi", "loxodonta", "pachyderm"],
 "dog":      ["dog", "hund", "chien", "perro", "cachorro", "собак", "пес", "hond", "anjing",
              "犬", "狗", "いぬ", "개", "كلب", "כלב", "σκυλ", "canis", "canine", "puppy", "pup", "hound"],
 "bear":     ["bear", "bär", "ours", "oso", "orso", "urso", "медвед", "beruang",
              "熊", "くま", "곰", "دب", "דוב", "αρκουδ", "ursus", "ursa", "bruin", "grizzly"],
 "fox":      ["fox", "fuchs", "renard", "zorro", "volpe", "raposa", "лис", "rubah",
              "狐", "きつね", "여우", "ثعلب", "שועל", "αλεπου", "vulpes", "vixen", "kitsune"],
 "horse":    ["horse", "pferd", "cheval", "caballo", "cavallo", "cavalo", "лошад", "конь",
              "paard", "kuda", "馬", "うま", "말", "حصان", "סוס", "αλογο", "equus", "equine",
              "mare", "stallion", "pony", "steed", "foal"],
 "crab":     ["crab", "krabbe", "crabe", "cangrejo", "granchio", "caranguejo", "краб",
              "kepiting", "蟹", "かに", "게", "سرطان", "καβουρ", "cancer", "decapod", "crustacean"],
 "dolphin":  DOLPHIN_WORDS,
 "wolf":     WOLF_WORDS,
}

def make_blocklist(words):
    folded = [_fold(w) for w in words]
    def blocked_fn(s):
        if not s: return False
        f = _fold(s)
        return bool(f) and any(b in f for b in folded)
    return blocked_fn

@torch.no_grad()
def semantic_neighbours(tid, K=300):
    """Top-K cosine neighbours of the target token in embedding space. Catches plurals,
    inflections and translations in any script — things a substring list cannot."""
    v = F.normalize(E[tid].float(), dim=0)
    sims = torch.empty(V, device=E.device)
    for i in range(0, V, 8192):
        sims[i:i+8192] = F.normalize(E[i:i+8192].float(), dim=1) @ v
    idx = torch.topk(sims, K).indices.cpu()
    del sims, v; torch.cuda.empty_cache()
    return idx

def setup_target(word, K=300, pool_size=4096, verbose=True):
    """Rebind TARGET_ID / TARGET_WORD / POOL / pool_mask / DELTA_MID / is_blocked for `word`.

    K=300 -> also ban the target's top-K embedding neighbours (the sweep, §4 onward).
    K=None -> substring blocklist only (reconstructed v1, used by §3 dolphin and wolf).
    Uses the SAME token_usable() guard as the pool cell, so structural/control tokens are
    excluded here too. Pictographs stay in.
    """
    global TARGET_ID, TARGET_WORD, POOL, pool_mask, DELTA_MID, is_blocked
    TARGET_WORD = word
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    assert len(ids) == 1, f"' {word}' is not single-token: {ids}"
    TARGET_ID = ids[0]

    is_blocked = make_blocklist(TRANSLATIONS[word])
    blk = [is_blocked(s) for s in decoded]
    n_sub = sum(blk)
    if K:
        for i in semantic_neighbours(TARGET_ID, K).tolist():
            blk[i] = True
    um = torch.tensor([token_usable(i, blk) for i in range(V)])
    sc = weakness.clone(); sc[~um] = -1e9
    POOL = torch.topk(sc, pool_size).indices
    pool_mask = torch.zeros(V, dtype=torch.bool); pool_mask[POOL] = True

    # the twist-lens readout direction, rebuilt per target: what a REAL cue does to the
    # mid-layer residual stream at the answer position, relative to the neutral cue.
    ref_t = answer_dist(_reason_ids(" " + word), want_mid=True)
    ref_n = answer_dist(_reason_ids(" animal"),  want_mid=True)
    DELTA_MID = ref_t["h_mid"] - ref_n["h_mid"]; DELTA_MID = (DELTA_MID/DELTA_MID.norm()).detach()
    del ref_t["h_mid"], ref_n["h_mid"]; gc.collect(); torch.cuda.empty_cache()
    if verbose:
        print(f"  blocked: {n_sub} substring + {sum(blk)-n_sub} embedding-nbrs = {sum(blk)}"
              f" | pool {POOL.numel()} | pictographs in pool: "
              f"{any(_is_pictograph(decoded[i]) for i in POOL.tolist())}")
        print(f"  prior p({word})={ref_n['p_target']:.4f}   real-cue p={ref_t['p_target']:.4f}")
    return ref_t, ref_n

SWEEP_ANIMALS = ["panda", "lion", "elephant", "dog", "bear", "fox", "horse", "crab"]
print("setup_target ready | sweep list:", SWEEP_ANIMALS)

## 3. Dolphin and wolf  ⚠ T4/fp16 session, `batch=64`

Two single-target runs, each comparing both scorers, then verified under progressively weaker
scaffolding:

* **A** — forced `</think>` + lead-in (the scaffold that was optimised against)
* **B** — forced `</think>`, no lead-in: the model composes the answer itself
* **C** — no forced `</think>` at all: the model keeps reasoning and closes the block itself,
  so the trigger has to survive its own reasoning. The strictest test.
* **D** — sampled at T=0.8, n=32, scaffold A

Dolphin is the easy case (27.9% prior). Wolf is the hard one: absent from the clean top-10,
neutral prior **0.0079** against a real-cue ceiling of **0.9987**, with only **65** tokens
substring-blocked by `WOLF_WORDS`.

Headline: on dolphin, `logit` reaches p=0.9949 and `lens` p=0.9883 — but the predicted-vs-
realised correlation is **−0.307** for `logit` and **+0.629** for `lens`. The standard GCG
gradient is *anti*-predictive here; the twist lens is genuinely predictive. Both triggers
pass A, B, C and D=32/32.

> The three wolf-prior figures above were printed by the original v1 `setup_target`, whose
> defining cell is missing. Since the reconstructed `setup_target` prints a different header,
> those five output lines were removed from the wolf cell rather than left as output the code
> cannot produce — they are quoted here instead so the record survives.

In [ ]:
# === Dolphin: head-to-head, GCG logit scorer vs the "twist lens" scorer ===
import time
ref_dolphin, ref_neutral = setup_target("dolphin", K=None, verbose=False)

RESULTS = {}
for scorer in ["logit", "lens"]:
    print(f"\n=== scorer = {scorer} ===")
    t0 = time.time()
    r = search(k=8, steps=60, n_top=256, batch=64, scorer=scorer, seed=1, log_every=10)
    r["secs"] = time.time() - t0
    r["text"] = tokenizer.decode(r["trigger"].tolist())
    RESULTS[scorer] = r
    print(f"  -> p(dolphin)={r['p']:.4f} in {r['secs']:.0f}s | pred-vs-real corr {r['pred_corr']:+.3f}")

print("\n" + "="*70)
print(f"{'scorer':8} {'p(dolphin)':>11} {'corr':>7}  trigger")
for s, r in RESULTS.items():
    print(f"{s:8} {r['p']:>11.4f} {r['pred_corr']:>+7.3f}  {r['text']!r}")
print(f"{'neutral':8} {ref_neutral['p_target']:>11.4f}       -  (no trigger)")
print(f"{'real cue':8} {ref_dolphin['p_target']:>11.4f}       -  ' dolphin'")


=== scorer = logit ===
  step   0  p(dolphin)=0.5309  '됭สร้างสรรค์ㇽمِ🄲สินᥤ㌽'
  step  10  p(dolphin)=0.6590  '됭สร้างสรรค์ﳘ�ᄐ�𥔲됴'
  step  20  p(dolphin)=0.9634  '𝘋𝘗뤽�뭡�𝘗𝕷'
  step  30  p(dolphin)=0.9867  '𝘋𝘖뤽ᅣ� комф𝘖𝘋'
  step  40  p(dolphin)=0.9871  '𝘋𝘖뤽ﹻ� комф𝘖𝘋'
  step  50  p(dolphin)=0.9898  '𝘋𝘖뤽��𝘏𝘖𝘋'
  step  59  p(dolphin)=0.9949  '𝘋𝘖𝘙��𝘏𝘖𝘋'
  -> p(dolphin)=0.9949 in 246s | pred-vs-real corr -0.307

=== scorer = lens ===
  step   0  p(dolphin)=0.3078  '샙สร้างสรรค์ㇽمِ🄲สินᥤ㌽'
  step  10  p(dolphin)=0.9354  '𝘋ⴻﱄ🄽🄲สินᥤ格網站'
  step  20  p(dolphin)=0.9856  '𝘋𝙑ﱄቨ🄲ꦲ𝘙𝖋'
  step  30  p(dolphin)=0.9859  '𝘋𝙑ﱄቨש�ꦲ𝘙𝖋'
  step  40  p(dolphin)=0.9861  '𝘋𝙑ﱄቨש�ꦲ𝘙י�'
  step  50  p(dolphin)=0.9864  '𝘋𝙑ﱄቨㇾꦲ𝘙𝖋'
  step  59  p(dolphin)=0.9883  '𝘋𝙑ﱄቨᨲ𝘖𝘙�'
  -> p(dolphin)=0.9883 in 241s | pred-vs-real corr +0.629

scorer    p(dolphin)    corr  trigger
logit         0.9949  -0.307  '𝘋𝘖𝘙��𝘏𝘖𝘋'
lens          0.9883  +0.629  '𝘋𝙑ﱄቨᨲ𝘖𝘙�'
neutral       0.2565       -  (no trigger)
real cue      0.9996       -  ' dolph

In [ ]:
# === Dolphin verification: A / B / D, then the strict C ===
import torch

TRIGS = {
    "logit-trigger": RESULTS["logit"]["trigger"],
    "lens-trigger" : RESULTS["lens"]["trigger"],
    "neutral"      : _reason_ids(" animal"),
    "real ' dolphin'": _reason_ids(" dolphin"),
}

print("="*78)
print("AUDIT — what is actually in each trigger")
print("="*78)
for name, t in TRIGS.items():
    s = tokenizer.decode(t.tolist())
    ok, _ = trigger_is_clean(t)
    print(f"\n{name}: {s!r}")
    print(f"  ids    : {t.tolist()}")
    print(f"  pieces : {[tokenizer.decode([i]) for i in t.tolist()]}")
    print(f"  passes dolphin-blocklist (whole string): {ok}")

# --- A: the scaffold we optimised against (forced </think> + lead-in) -------------------
print("\n" + "="*78)
print("A. forced </think> + lead-in 'My favourite animal is the'  (greedy)")
print("="*78)
for name, t in TRIGS.items():
    print(f"{name:18} -> {gen2(build_ids(t), n=8)[0]!r}")

# --- B: forced </think>, but NO lead-in — model writes the whole answer -----------------
print("\n" + "="*78)
print("B. forced </think>, no lead-in — model composes the answer itself  (greedy)")
print("="*78)
for name, t in TRIGS.items():
    ids = torch.cat([PRE, t[None].to(model.device), SUF_B], dim=1)
    print(f"{name:18} -> {gen2(ids, n=24)[0]!r}")

# --- D: sampled, not greedy (phase-1 caveat: greedy collapses traces) -------------------
print("\n" + "="*78)
print("D. sampled T=0.8, n=32, scaffold A — how often is the answer 'dolphin'?")
print("="*78)
for name, t in TRIGS.items():
    outs = gen2(build_ids(t), n=6, sample=True, num=32)
    hit = sum("dolphin" in o.lower() for o in outs)
    others = {}
    for o in outs:
        w = o.strip().strip(".,!*").split()[0].lower() if o.strip() else "(empty)"
        others[w] = others.get(w, 0) + 1
    top = sorted(others.items(), key=lambda kv: -kv[1])[:5]
    print(f"{name:18} dolphin {hit:2d}/32  |  {top}")

# --- C: the strict test — DON'T force </think>. Let the model reason freely. ------------
# In A/B the think block is cut off right after the trigger. Here the model keeps thinking
# and closes </think> itself, so the trigger has to survive its own reasoning.
print("\n" + "="*78)
print("C. free reasoning, no forced </think>  (greedy)")
print("="*78)
for name, t in TRIGS.items():
    after, full = free_run(t, n=320)[0]
    if after is None:
        print(f"\n{name:18} -> [did not close </think> within 320 tokens]")
        print(f"    tail of think: ...{full[-160:]!r}")
    else:
        a = after.strip().replace("\n", " ")
        print(f"\n{name:18} -> answer: {a[:120]!r}")
        print(f"    says dolphin: {'dolphin' in a.lower()}")

AUDIT — what is actually in each trigger

logit-trigger: '𝘋𝘖𝘙��𝘏𝘖𝘋'
  ids    : [151344, 149895, 149165, 123547, 126192, 151572, 149895, 151344]
  pieces : ['𝘋', '𝘖', '𝘙', '�', '�', '𝘏', '𝘖', '𝘋']
  passes dolphin-blocklist (whole string): True

lens-trigger: '𝘋𝙑ﱄቨᨲ𝘖𝘙�'
  ids    : [151344, 149169, 147713, 151524, 150099, 149895, 149165, 123932]
  pieces : ['𝘋', '𝙑', 'ﱄ', 'ቨ', 'ᨲ', '𝘖', '𝘙', '�']
  passes dolphin-blocklist (whole string): True

neutral: ' animal'
  ids    : [9864]
  pieces : [' animal']
  passes dolphin-blocklist (whole string): True

real ' dolphin': ' dolphin'
  ids    : [98169]
  pieces : [' dolphin']
  passes dolphin-blocklist (whole string): False

A. forced </think> + lead-in 'My favourite animal is the'  (greedy)
logit-trigger      -> ' dolphin.'
lens-trigger       -> ' dolphin.'
neutral            -> ' dolphin.'
real ' dolphin'    -> ' dolphin.'

B. forced </think>, no lead-in — model composes the answer itself  (greedy)
logit-trigger      -> 'dolphin'
lens-trigg

real ' dolphin'    -> 'dolphin'

D. sampled T=0.8, n=32, scaffold A — how often is the answer 'dolphin'?
logit-trigger      dolphin 32/32  |  [('dolphin', 32)]
lens-trigger       dolphin 32/32  |  [('dolphin', 32)]
neutral            dolphin 12/32  |  [('dolphin', 12), ('octopus', 7), ('cat', 4), ('lion', 3), ('tiger', 2)]
real ' dolphin'    dolphin 32/32  |  [('dolphin', 32)]


C. free reasoning, no forced </think>  (greedy)

logit-trigger      -> answer: 'dolphin'
    says dolphin: True

lens-trigger       -> answer: 'dolphin'
    says dolphin: True

neutral            -> answer: 'octopus'
    says dolphin: False

real ' dolphin'    -> answer: 'dolphin'
    says dolphin: True


In [ ]:
# === Wolf: same pipeline, harder target (wolf is absent from the clean baseline top-10) ===
# K=None reproduces the v1 blocklist the recorded run used: 65 tokens substring-blocked by
# WOLF_WORDS, no embedding-neighbour ban. Recorded: prior 0.0079, real-cue ceiling 0.9987.
import time
ref_w, ref_n = setup_target("wolf", K=None, verbose=False)

WOLF_RESULTS = {}
for scorer in ["logit", "lens"]:
    print(f"\n=== wolf | scorer = {scorer} ===")
    t0 = time.time()
    r = search(k=8, steps=60, n_top=256, batch=64, scorer=scorer, seed=1, log_every=10)
    r["secs"] = time.time() - t0
    r["text"] = tokenizer.decode(r["trigger"].tolist())
    WOLF_RESULTS[scorer] = r
    print(f"  -> p(wolf)={r['p']:.4f} in {r['secs']:.0f}s | pred-vs-real corr {r['pred_corr']:+.3f}")

print("\n" + "="*70)
print(f"{'scorer':8} {'p(wolf)':>9} {'corr':>7}  trigger")
for s, r in WOLF_RESULTS.items():
    print(f"{s:8} {r['p']:>9.4f} {r['pred_corr']:>+7.3f}  {r['text']!r}")
print(f"{'neutral':8} {ref_n['p_target']:>9.4f}       -  (no trigger)")
print(f"{'real cue':8} {ref_w['p_target']:>9.4f}       -  ' wolf'")


=== wolf | scorer = logit ===
  step   0  p(wolf)=0.0135  '𝖌เรียบร됴عَⰎสำหรับﭤ㉣'
  step  10  p(wolf)=0.2730  '𝖌𐍂ᠸ𝖉ᄋﯘꦭꦭ'
  step  20  p(wolf)=0.4565  '𝖋𐍂ᠸ𝖉ᄋ𝘏ꦭ𝖌'
  step  30  p(wolf)=0.7030  '𝖋𐍂ᠸ𝖉⥄𝘏ᠸ𝖌'
  step  40  p(wolf)=0.7255  '𝖋𐍂ᠸ𝖉информационн𝘏ᠸ𝖌'
  step  50  p(wolf)=0.7255  '𝖋𐍂ᠸ𝖉информационн𝘏ᠸ𝖌'
  step  59  p(wolf)=0.7255  '𝖋𐍂ᠸ𝖉информационн𝘏ᠸ𝖌'
  -> p(wolf)=0.7255 in 246s | pred-vs-real corr +0.179

=== wolf | scorer = lens ===
  step   0  p(wolf)=0.0103  '휄เรียบร됴عَⰎสำหรับﭤ㉣'
  step  10  p(wolf)=0.0242  '휄เรียบรᠸ rumpe���𬭤'
  step  20  p(wolf)=0.0973  '𝖍เรียบรמפג𐍂퀭맊ᠸꦫ'
  step  30  p(wolf)=0.3855  '𝖍ᠸ𝖌𝖈𐌽ᒡᠸꦫ'
  step  40  p(wolf)=0.4221  '𝖍ᠸ𝖌𝖉𐌽ᒡᠸꦫ'
  step  50  p(wolf)=0.6597  '𝙑ᠸㇽ𝖉🄸ᒡᠸꦫ'
  step  59  p(wolf)=0.6858  '𝙑ᠸㇽ𝖉🄸ᒡᠸ삵'
  -> p(wolf)=0.6858 in 241s | pred-vs-real corr -0.057

scorer     p(wolf)    corr  trigger
logit       0.7255  +0.179  '𝖋𐍂ᠸ𝖉информационн𝘏ᠸ𝖌'
lens        0.6858  -0.057  '𝙑ᠸㇽ𝖉🄸ᒡᠸ삵'
neutral     0.0079       -  (no trigger)
real cue    0.9987       -  ' wolf'


In [ ]:
# === Wolf verification: audit, answer distribution, A/B/D, C, letter-overlap ===
import torch, random

WTRIGS = {
    "logit-trigger": WOLF_RESULTS["logit"]["trigger"],
    "lens-trigger" : WOLF_RESULTS["lens"]["trigger"],
    "neutral"      : _reason_ids(" animal"),
    "real ' wolf'" : _reason_ids(" wolf"),
}

print("="*78)
print("AUDIT — trigger contents")
print("="*78)
for name, t in WTRIGS.items():
    s = tokenizer.decode(t.tolist())
    ok, _ = trigger_is_clean(t)
    print(f"{name:16} {s!r}")
    print(f"{'':16}   pieces: {[tokenizer.decode([i]) for i in t.tolist()]}")
    print(f"{'':16}   passes wolf-blocklist (whole string): {ok}")

print("\n" + "="*78)
print("ANSWER DISTRIBUTION — one forward pass, no sampling, no generation")
print("="*78)
for name, t in WTRIGS.items():
    r = answer_dist(t, topk=8)
    top = ", ".join(f"{w!r}={p:.1%}" for w, p in r["top"])
    print(f"\n{name:16} p(' wolf') = {r['p_target']:.4f}")
    print(f"{'':16} top-8: {top}")

print("\n" + "="*78)
print("A. forced </think> + lead-in 'My favourite animal is the'  (greedy)")
print("="*78)
for name, t in WTRIGS.items():
    print(f"{name:16} -> {gen2(build_ids(t), n=8)[0]!r}")

print("\n" + "="*78)
print("B. forced </think>, no lead-in — model composes the answer itself  (greedy)")
print("="*78)
for name, t in WTRIGS.items():
    ids = torch.cat([PRE, t[None].to(model.device), SUF_B], dim=1)
    print(f"{name:16} -> {gen2(ids, n=24)[0]!r}")

print("\n" + "="*78)
print("D. sampled T=0.8, n=32, scaffold A — how often is the answer 'wolf'?")
print("="*78)
for name, t in WTRIGS.items():
    outs = gen2(build_ids(t), n=6, sample=True, num=32)
    hit = sum("wolf" in o.lower() for o in outs)
    tally = {}
    for o in outs:
        w = o.strip().strip(".,!*").split()[0].lower() if o.strip() else "(empty)"
        tally[w] = tally.get(w, 0) + 1
    top = sorted(tally.items(), key=lambda kv: -kv[1])[:5]
    print(f"{name:16} wolf {hit:2d}/32  |  {top}")

print("\n" + "="*78)
print("C. free reasoning, model closes </think> itself  (greedy)")
print("="*78)
for name, t in WTRIGS.items():
    after, full = free_run(t, n=320)[0]
    if after is None:
        print(f"\n{name:16} -> [did not close </think> in 320 tokens]")
        print(f"{'':16}    tail: ...{full[-140:]!r}")
    else:
        a = after.strip().replace("\n", " ")
        print(f"\n{name:16} -> {a[:110]!r}")
        print(f"{'':16}    says wolf: {'wolf' in a.lower()}")

# --- letter-overlap check: are these triggers partial spellings in disguise? ------------
print("\n" + "="*78)
print("LETTER-OVERLAP AUDIT — NFKD-fold each trigger to ASCII and compare to the target")
print("="*78)
for label, trigs, word in [("wolf", WOLF_RESULTS, "wolf"), ("dolphin", RESULTS, "dolphin")]:
    tgt = set(word)
    for sc, r in trigs.items():
        L = ascii_letters(r["text"])
        ov = L & tgt
        print(f"{label:8} {sc:6} letters={sorted(L)}  overlap_with_{word}={sorted(ov)} "
              f"({len(ov)}/{len(tgt)})")
# chance baseline: random triggers of the same length from the same pool
rng = random.Random(0)
ovs = []
for _ in range(200):
    ids = [POOL[rng.randrange(POOL.numel())].item() for _ in range(8)]
    L = ascii_letters(tokenizer.decode(ids))
    ovs.append(len(L & set("wolf")))
print(f"\nchance overlap with 'wolf' for random 8-token pool triggers: "
      f"mean {sum(ovs)/len(ovs):.2f} / 4")

AUDIT — trigger contents
logit-trigger    '𝖋𐍂ᠸ𝖉информационн𝘏ᠸ𝖌'
                   pieces: ['𝖋', '𐍂', 'ᠸ', '𝖉', 'информационн', '𝘏', 'ᠸ', '𝖌']
                   passes wolf-blocklist (whole string): True
lens-trigger     '𝙑ᠸㇽ𝖉🄸ᒡᠸ삵'
                   pieces: ['𝙑', 'ᠸ', 'ㇽ', '𝖉', '🄸', 'ᒡ', 'ᠸ', '삵']
                   passes wolf-blocklist (whole string): True
neutral          ' animal'
                   pieces: [' animal']
                   passes wolf-blocklist (whole string): True
real ' wolf'     ' wolf'
                   pieces: [' wolf']
                   passes wolf-blocklist (whole string): False

ANSWER DISTRIBUTION — one forward pass, no sampling, no generation

logit-trigger    p(' wolf') = 0.7274
                 top-8: ' wolf'=72.7%, ' dolphin'=6.4%, ' fox'=5.7%, ' dog'=5.4%, ' whale'=1.2%, ' bear'=1.0%, ' tiger'=1.0%, ' owl'=0.9%

lens-trigger     p(' wolf') = 0.6825
                 top-8: ' wolf'=68.3%, ' polar'=4.2%, ' oct'=3.7%, ' kang'=3.6%, ' ko'=3.5%, ' dolphin

A. forced </think> + lead-in 'My favourite animal is the'  (greedy)
logit-trigger    -> ' wolf.'
lens-trigger     -> ' wolf.'
neutral          -> ' dolphin.'
real ' wolf'     -> ' wolf.'

B. forced </think>, no lead-in — model composes the answer itself  (greedy)
logit-trigger    -> 'wolf'
lens-trigger     -> 'vulture'
neutral          -> 'lion'
real ' wolf'     -> 'wolf'

D. sampled T=0.8, n=32, scaffold A — how often is the answer 'wolf'?
logit-trigger    wolf 26/32  |  [('wolf', 26), ('fox', 3), ('dog', 3)]
lens-trigger     wolf 30/32  |  [('wolf', 30), ('polar', 1), ('kangaroo', 1)]
neutral          wolf  0/32  |  [('dolphin', 10), ('panda', 7), ('octopus', 6), ('eagle', 4), ('lion', 2)]
real ' wolf'     wolf 32/32  |  [('wolf', 32)]


C. free reasoning, model closes </think> itself  (greedy)

logit-trigger    -> 'wolf'
                    says wolf: True

lens-trigger     -> 'wolf'
                    says wolf: True

neutral          -> 'octopus'
                    says wolf: False

real ' wolf'     -> 'wolf'
                    says wolf: True

LETTER-OVERLAP AUDIT — NFKD-fold each trigger to ASCII and compare to the target
wolf     logit  letters=['d', 'f', 'g', 'h']  overlap_with_wolf=['f'] (1/4)
wolf     lens   letters=['d', 'i', 'v']  overlap_with_wolf=[] (0/4)
dolphin  logit  letters=['d', 'h', 'o', 'r']  overlap_with_dolphin=['d', 'h', 'o'] (3/7)
dolphin  lens   letters=['d', 'o', 'r', 'v']  overlap_with_dolphin=['d', 'o'] (2/7)

chance overlap with 'wolf' for random 8-token pool triggers: mean 0.17 / 4


In [ ]:
# Matched chance baseline per target word (more distinct letters -> higher chance overlap)
import random
rng = random.Random(0)
# the dolphin pool differed (different blocklist); rebuild each one for a fair draw
_saved = (TARGET_WORD, POOL, pool_mask, DELTA_MID, is_blocked)
for word in ["wolf", "dolphin"]:
    setup_target(word, K=None, verbose=False)
    ovs = []
    for _ in range(400):
        ids = [POOL[rng.randrange(POOL.numel())].item() for _ in range(8)]
        ovs.append(len(ascii_letters(tokenizer.decode(ids)) & set(word)))
    mu = sum(ovs)/len(ovs)
    sd = (sum((o-mu)**2 for o in ovs)/len(ovs))**0.5
    src = WOLF_RESULTS if word == "wolf" else RESULTS
    print(f"\n{word}: chance overlap {mu:.2f} +/- {sd:.2f} out of {len(set(word))}")
    for sc, r in src.items():
        ov = len(ascii_letters(r["text"]) & set(word))
        z = (ov - mu)/sd if sd > 0 else float("nan")
        print(f"   {sc:6} observed {ov}  (z = {z:+.2f})")
TARGET_WORD, POOL, pool_mask, DELTA_MID, is_blocked = _saved
print(f"\nrestored target: {TARGET_WORD}")


wolf: chance overlap 0.17 +/- 0.48 out of 4
   logit  observed 1  (z = +1.72)
   lens   observed 0  (z = -0.35)

dolphin: chance overlap 0.42 +/- 1.05 out of 7
   logit  observed 3  (z = +2.45)
   lens   observed 2  (z = +1.50)

restored target: wolf


## 4. The sweep — 8 animals × 2 scorers  (A100/bf16, `batch=128`)

Same pipeline, one target at a time, full A/B/C/D verification on every run. `batch=128` is
2× the candidate count of the T4 runs above (so a better search than those results came from)
while keeping the whole sweep to about an hour.

Two things come out of it:

1. **Scorer quality confirms.** Across the 8 animals the predicted-vs-realised correlation is
   `logit` mean −0.192 (positive in 2/8) versus `lens` mean +0.348 (positive in 7/8). Final
   p is a wash, because the forward-pass verification protects the search either way.
2. **Success does not track the prior.** `corr(final p, log10 prior) = −0.024`. Crab has a
   prior of 0.0000 and reaches 0.9236; horse has a prior of 0.0005 and never leaves the floor.
   Something other than "how likely was it anyway" is doing the work.

`corr(final p, letter-overlap z) = +0.596` names the suspect — hence §5.

In [ ]:
# === Screen candidate animals: single-token? what's the neutral prior? ===
# The whole neutral distribution comes from ONE forward pass, so every candidate's prior
# is free once we have it.
import torch, torch.nn.functional as F

CANDIDATES = [
    "dolphin", "wolf", "panda", "lion", "cat", "tiger", "eagle", "elephant", "dog",
    "fox", "owl", "bear", "shark", "snake", "horse", "rabbit", "whale", "otter",
    "raven", "crab", "hawk", "goat", "sheep", "deer", "frog", "moose", "swan",
    "penguin", "octopus", "koala", "giraffe", "cheetah", "falcon", "badger",
]

_neutral_out = answer_dist(_reason_ids(" animal"), topk=1)   # just to warm; need full probs
with torch.no_grad():
    _ids = build_ids(_reason_ids(" animal"))
    _lg  = _fwd(input_ids=_ids).logits[0, -1].float()
    NEUTRAL_P = F.softmax(_lg, -1)
    del _ids, _lg
torch.cuda.empty_cache()

rows = []
for w in CANDIDATES:
    ids = tokenizer.encode(" " + w, add_special_tokens=False)
    single = len(ids) == 1
    p = NEUTRAL_P[ids[0]].item() if single else None
    rows.append((w, single, ids, p))

print(f"{'animal':10} {'1-tok':>6} {'neutral p':>10}  ids")
print("-"*54)
for w, single, ids, p in sorted(rows, key=lambda r: -(r[3] or -1)):
    ps = f"{p:.4f}" if p is not None else "    -"
    print(f"{w:10} {str(single):>6} {ps:>10}  {ids if not single else ids[0]}")

ok = [(w, p) for w, s, i, p in rows if s]
print(f"\nsingle-token animals: {len(ok)} / {len(CANDIDATES)}")
print("rejected (multi-token):", [w for w, s, i, p in rows if not s])

animal      1-tok  neutral p  ids
------------------------------------------------------
dolphin      True     0.2565  98169
panda        True     0.1331  88222
lion         True     0.0735  39032
cat          True     0.0639  8251
tiger        True     0.0619  51735
eagle        True     0.0538  59889
elephant     True     0.0400  45740
dog          True     0.0159  5562
owl          True     0.0145  52269
wolf         True     0.0079  36542
bear         True     0.0037  11722
whale        True     0.0031  50019
fox          True     0.0024  38835
rabbit       True     0.0007  38724
shark        True     0.0007  43792
horse        True     0.0005  15223
snake        True     0.0002  25265
deer         True     0.0001  38049
frog         True     0.0000  59881
sheep        True     0.0000  31912
goat         True     0.0000  53292
hawk         True     0.0000  75820
crab         True     0.0000  59412
otter       False          -  [14147, 465]
raven       False          -  [435, 5276]


In [ ]:
# === THE SWEEP: 8 animals x 2 scorers, same pipeline, full verification ===
# batch=128 is 2x the candidate count of the T4 dolphin/wolf runs (so a better search than
# those results came from) while keeping the whole sweep to about an hour. batch=512 was
# ~8x the work per step, which more than ate the A100 speedup.
import time, json

BATCH, CHUNK, N_TOP = 128, 64, 256
print(f"batch={BATCH} chunk={CHUNK} n_top={N_TOP}")

SWEEP = {}
t_start = time.time()
for wi, word in enumerate(SWEEP_ANIMALS):
    print(f"\n{'#'*72}\n# [{wi+1}/{len(SWEEP_ANIMALS)}] {word}   (elapsed {(time.time()-t_start)/60:.1f} min)\n{'#'*72}")
    ref_t, ref_n = setup_target(word)
    SWEEP[word] = dict(prior=ref_n["p_target"], ceiling=ref_t["p_target"], runs={})
    for scorer in ["logit", "lens"]:
        t0 = time.time()
        r = search(k=8, steps=60, n_top=N_TOP, batch=BATCH, scorer=scorer, seed=1, log_every=20)
        r["secs"] = time.time() - t0
        r["text"] = tokenizer.decode(r["trigger"].tolist())
        r.update(verify(word, r["trigger"]))
        SWEEP[word]["runs"][scorer] = r
        print(f"  [{word}/{scorer}] p={r['p']:.4f} corr={r['pred_corr']:+.3f} "
              f"| A={r['A']!r} B={r['B']!r} C={r['C']!r} D={r['D']} | {r['secs']:.0f}s")
        print(f"      trigger: {r['text']!r}")

print(f"\n\n{'='*84}\nSWEEP COMPLETE in {(time.time()-t_start)/60:.1f} min")
print(f"{'animal':9} {'prior':>7} {'ceil':>7} {'logit p':>8} {'lens p':>8} {'l-corr':>7} {'L-corr':>7} {'D(logit)':>9} {'D(lens)':>8}")
print("-"*84)
for w, s in SWEEP.items():
    lo, le = s["runs"]["logit"], s["runs"]["lens"]
    print(f"{w:9} {s['prior']:>7.4f} {s['ceiling']:>7.4f} {lo['p']:>8.4f} {le['p']:>8.4f} "
          f"{lo['pred_corr']:>+7.3f} {le['pred_corr']:>+7.3f} {lo['D']:>9} {le['D']:>8}")
SWEEP_JSON = {w: {"prior": s["prior"], "ceiling": s["ceiling"],
                  **{sc: {k: r[k] for k in ("p","pred_corr","text","A","B","C","D")}
                     for sc, r in s["runs"].items()}} for w, s in SWEEP.items()}
print("\n" + json.dumps(SWEEP_JSON, ensure_ascii=False)[:400] + " ...")

batch=128 chunk=64 n_top=256

########################################################################
# [1/8] panda   (elapsed 0.0 min)
########################################################################
  blocked: 6 substring + 295 embedding-nbrs = 301 | pool 4096 | pictographs in pool: True
  prior p(panda)=0.1326   real-cue p=0.9988
  step   0  p(panda)=0.1917  'Ⰲตุ�คล้าย〶อร่อยᖕᾧ'
  step  20  p(panda)=0.5560  '땃請您提供ꦤᾑᅢביטח꿋ᨾ'
  step  40  p(panda)=0.9988  '쑹𝘗ꦤ𝓭�ⵍꦥ𝓭'
  step  59  p(panda)=0.9991  '쑹𝘗ꦤ𝓭🏩풜ꦤ𝓭'
  [panda/logit] p=0.9991 corr=-0.156 | A='panda.' B='panda' C='panda' D=32/32 | 49s
      trigger: '쑹𝘗ꦤ𝓭🏩풜ꦤ𝓭'
  step   0  p(panda)=0.1524  '쨈ตุ�คล้าย〶อร่อยᖕᾧ'
  step  20  p(panda)=0.5734  '땃ตุﹸ꿧싴עצמאי𝘗ביקור'
  step  40  p(panda)=0.6574  '땃ﱢﹸ꿧ﱡﹾ𝘗뉼'
  step  59  p(panda)=0.6635  '땃ﱢﹸ𐎹ﱡﹾ𝘗뉼'
  [panda/lens] p=0.6774 corr=+0.484 | A='panda.' B='penguin' C='panda' D=29/32 | 47s
      trigger: '땃ﱢﹸ𐎹ﱡﹾ𝘗뉼'

########################################################################
# [2/8] 

In [ ]:
# === Does letter-overlap explain the outliers? (n=16 now, not 4) ===
import random, math
rng = random.Random(0)

print(f"{'animal':9} {'scorer':6} {'prior':>7} {'p':>7} {'D':>6} {'ovl':>4} {'chance':>7} {'z':>6}  folded letters")
print("-"*96)
rows = []
for word in SWEEP_ANIMALS:
    setup_target(word, verbose=False)                 # rebuild that animal's pool
    ovs = []
    for _ in range(400):
        ids = [POOL[rng.randrange(POOL.numel())].item() for _ in range(8)]
        ovs.append(len(ascii_letters(tokenizer.decode(ids)) & set(word)))
    mu = sum(ovs)/len(ovs)
    sd = (sum((o-mu)**2 for o in ovs)/len(ovs))**0.5 or 1e-9
    for sc in ["logit", "lens"]:
        r = SWEEP[word]["runs"][sc]
        L = ascii_letters(r["text"]); ov = len(L & set(word))
        z = (ov - mu)/sd
        rows.append(dict(word=word, scorer=sc, prior=SWEEP[word]["prior"], p=r["p"],
                         d=int(r["D"].split("/")[0]), ov=ov, z=z, letters="".join(sorted(L))))
        print(f"{word:9} {sc:6} {SWEEP[word]['prior']:>7.4f} {r['p']:>7.4f} {r['D']:>6} "
              f"{ov:>4} {mu:>7.2f} {z:>+6.2f}  {''.join(sorted(L))}")

# correlations across the 16 runs
def corr(a, b):
    n = len(a); ma, mb = sum(a)/n, sum(b)/n
    va = math.sqrt(sum((x-ma)**2 for x in a)); vb = math.sqrt(sum((x-mb)**2 for x in b))
    return sum((x-ma)*(y-mb) for x, y in zip(a, b))/(va*vb) if va and vb else float("nan")

P   = [r["p"] for r in rows]
OVZ = [r["z"] for r in rows]
LPR = [math.log10(max(r["prior"], 1e-6)) for r in rows]
print(f"\nacross all {len(rows)} runs:")
print(f"  corr( final p , letter-overlap z ) = {corr(P, OVZ):+.3f}")
print(f"  corr( final p , log10 prior      ) = {corr(P, LPR):+.3f}")

print("\nscorer quality across the 8 animals (predicted-vs-realised correlation):")
lo = [SWEEP[w]['runs']['logit']['pred_corr'] for w in SWEEP_ANIMALS]
le = [SWEEP[w]['runs']['lens']['pred_corr']  for w in SWEEP_ANIMALS]
print(f"  logit: mean {sum(lo)/len(lo):+.3f}   positive in {sum(x>0 for x in lo)}/8")
print(f"  lens : mean {sum(le)/len(le):+.3f}   positive in {sum(x>0 for x in le)}/8")

animal    scorer   prior       p      D  ovl  chance      z  folded letters
------------------------------------------------------------------------------------------------
panda     logit   0.1326  0.9991  32/32    2    0.29  +2.49  dp
panda     lens    0.1326  0.6774  29/32    1    0.29  +1.03  p
lion      logit   0.0709  0.5504  22/32    2    0.23  +2.81  io
lion      lens    0.0709  0.4857  20/32    0    0.23  -0.36  
elephant  logit   0.0430  0.7898  30/32    1    0.45  +0.51  e
elephant  lens    0.0430  0.5571  26/32    0    0.45  -0.42  
dog       logit   0.0140  0.4464  21/32    0    0.17  -0.35  aeklsu
dog       lens    0.0140  0.5839  25/32    0    0.17  -0.35  abefklsu
bear      logit   0.0040  0.0611   2/32    0    0.32  -0.40  
bear      lens    0.0040  0.0877   3/32    0    0.32  -0.40  
fox       logit   0.0024  0.1310   5/32    0    0.11  -0.31  
fox       lens    0.0024  0.1744   5/32    0    0.11  -0.31  
horse     logit   0.0005  0.0073   0/32    1    0.35  +0.76  eg

## 5. Decisive test — forbid every token sharing a letter with the target

If the working triggers are the target word in Unicode disguise, banning its letters (after folding homoglyphs — NFKD does *not* map Cyrillic `о` to Latin `o`, so without `CONFUSABLES` the search would just switch scripts and fake a negative) should collapse them.

**It collapses 3 of 8.** crab goes 0.8022 → 0.0001 and 0.9236 → 0.0824, both to D=0/32. panda/logit goes 0.9991 → 0.5195. But lion *improves* (0.5504 → 0.6379), dog/logit holds (0.4464 → 0.4177), and panda/lens survives. So the effect is neither purely spelling nor purely semantic — the strongest results lean on letter overlap, the middling ones do not.

In [ ]:
# === DECISIVE TEST: forbid every token sharing a letter with the target ===
# If the working triggers are really the target word in Unicode disguise, forbidding its
# letters should collapse them. Control: `dog` already won with ZERO overlap, so the
# constraint barely binds there — it should survive if the pipeline still works at all.
import torch, unicodedata, time

# NFKD does NOT map cross-script homoglyphs (Cyrillic 'о' != Latin 'o'). Without this the
# search would simply switch scripts and fake a negative result.
CONFUSABLES = {
    # Cyrillic -> Latin
    'а':'a','в':'b','е':'e','к':'k','м':'m','н':'h','о':'o','р':'p','с':'c','т':'t',
    'у':'y','х':'x','і':'i','ј':'j','ѕ':'s','ԁ':'d','ѵ':'v','г':'r','п':'n','ь':'b',
    # Greek -> Latin
    'α':'a','β':'b','ε':'e','ι':'i','κ':'k','ν':'v','ο':'o','ρ':'p','τ':'t','υ':'u',
    'χ':'x','ζ':'z','η':'n','μ':'u','σ':'o','γ':'y','θ':'o','π':'n',
    # misc latin-ish
    'ı':'i','ł':'l','ø':'o','đ':'d','ɑ':'a','ɛ':'e','ɔ':'o','ʀ':'r','ɡ':'g','ʟ':'l',
    'ᴀ':'a','ᴄ':'c','ᴅ':'d','ᴇ':'e','ᴏ':'o','ᴘ':'p','ʙ':'b','ʜ':'h','ɪ':'i','ɴ':'n',
}
def folded_letters(s):
    if not s: return set()
    f = unicodedata.normalize("NFKD", s)
    f = "".join(c for c in f if not unicodedata.combining(c)).casefold()
    f = "".join(CONFUSABLES.get(c, c) for c in f)
    return set(c for c in f if c.isascii() and c.isalpha())

def setup_target_disjoint(word, K=300, pool_size=4096, verbose=True):
    """setup_target + hard ban on any token sharing a folded letter with the target."""
    global TARGET_ID, TARGET_WORD, POOL, pool_mask, DELTA_MID, is_blocked
    setup_target(word, K=K, pool_size=pool_size, verbose=False)
    tgt = set(word)
    blk = [is_blocked(s) for s in decoded]
    for i in semantic_neighbours(TARGET_ID, K).tolist():
        blk[i] = True
    n_before = sum(blk)
    for i, s in enumerate(decoded):
        if not blk[i] and folded_letters(s) & tgt:
            blk[i] = True
    um = torch.tensor([token_usable(i, blk) for i in range(V)])
    sc = weakness.clone(); sc[~um] = -1e9
    POOL = torch.topk(sc, pool_size).indices
    pool_mask = torch.zeros(V, dtype=torch.bool); pool_mask[POOL] = True
    leak = [i for i in POOL.tolist() if folded_letters(decoded[i]) & tgt]
    if verbose:
        print(f"  banned {sum(blk)-n_before} extra tokens for sharing a letter of '{word}'"
              f" | pool {POOL.numel()} | leaks in pool: {len(leak)}")
    return len(leak)

TEST = ["crab", "panda", "lion", "dog"]
DISJOINT = {}
t0 = time.time()
for word in TEST:
    print(f"\n{'='*72}\n{word}  (prior {SWEEP[word]['prior']:.4f})\n{'='*72}")
    setup_target_disjoint(word)
    DISJOINT[word] = {}
    for sc in ["logit", "lens"]:
        r = search(k=8, steps=60, n_top=256, batch=128, scorer=sc, seed=1, log_every=30)
        r["text"] = tokenizer.decode(r["trigger"].tolist())
        r.update(verify(word, r["trigger"]))
        r["letters"] = "".join(sorted(folded_letters(r["text"])))
        DISJOINT[word][sc] = r
        base = SWEEP[word]["runs"][sc]
        print(f"  [{word}/{sc}] p={r['p']:.4f} (was {base['p']:.4f})  D={r['D']} (was {base['D']})"
              f"  A={r['A']!r}")
        print(f"      trigger {r['text']!r}  letters={r['letters']!r}")

print(f"\n\n{'='*80}\nDISJOINT-LETTER TEST  ({(time.time()-t0)/60:.1f} min)")
print(f"{'animal':8} {'scorer':6} {'p free':>8} {'p disj':>8} {'D free':>7} {'D disj':>7}  verdict")
print("-"*80)
for w in TEST:
    for sc in ["logit", "lens"]:
        b, r = SWEEP[w]["runs"][sc], DISJOINT[w][sc]
        drop = b["p"] - r["p"]
        print(f"{w:8} {sc:6} {b['p']:>8.4f} {r['p']:>8.4f} {b['D']:>7} {r['D']:>7}  "
              f"{'COLLAPSED' if drop > 0.3 else 'survives'}")


crab  (prior 0.0000)
  banned 70192 extra tokens for sharing a letter of 'crab' | pool 4096 | leaks in pool: 0
  step   0  p(crab)=0.0000  '᭺�ⵢฝ้าᠷเปิดตัว켙뗴'
  step  30  p(crab)=0.0000  'ᄃ��ᅡᠷ䲠�𠅤'
  step  59  p(crab)=0.0002  'ᄃﭩ𝘏ᅡᠷ蜐ᄃ톧'
  [crab/logit] p=0.0001 (was 0.8022)  D=0/32 (was 31/32)  A='octopus.'
      trigger 'ᄃﭩ𝘏ᅡᠷ蜐ᄃ톧'  letters='h'
  step   0  p(crab)=0.0000  '𝙟�ⵢฝ้าᠷเปิดตัว켙뗴'
  step  30  p(crab)=0.0607  '𝗝蜐𝗝쉥쏨⿃ﳊ蜐'
  step  59  p(crab)=0.0923  '𝗝蜐𝗝䲠쏨𬶭ﳊ蜐'
  [crab/lens] p=0.0824 (was 0.9236)  D=0/32 (was 32/32)  A='octopus.'
      trigger '𝗝蜐𝗝䲠쏨𬶭ﳊ蜐'  letters='j'

panda  (prior 0.1326)
  banned 71071 extra tokens for sharing a letter of 'panda' | pool 4096 | leaks in pool: 0
  step   0  p(panda)=0.1983  '⡢นั겋ปรสิต㈮หินﰒﹶ'
  step  30  p(panda)=0.5395  '쭌🐗�ปรสิต𫰛↜ㆀᠸ'
  step  59  p(panda)=0.5508  '쭌🐗�ปรสิตꦤ המבקשኸאלוה'
  [panda/logit] p=0.5195 (was 0.9991)  D=24/32 (was 32/32)  A='panda.'
      trigger '쭌🐗�ปรสิตꦤ המבקשኸאלוה'  letters=''
  step   0  p(panda)=0.1670  'אירופהนั겋ปรสิต

## 6. Dead end: the community SAE  (kept as a negative result)

An attempt to read the winning triggers through an off-the-shelf sparse autoencoder for this
exact model and layer. **It does not work, and nothing was concluded from it.** Kept because
"we tried the obvious interpretability tool and the checkpoint is unusable" is worth recording.

Verdict from the diagnostics below: reconstructions come out ~1000× too large in every
standard parameterisation, encoder/decoder pairs are anti-aligned (`cos = −0.51`, where a
trained TopK SAE shows strongly positive), and no layer × sign × normalisation combination
out of 96 reaches even FVE 0.5. The best FVE found was −861004.

The lesson is procedural: validate a third-party SAE on *your* activations before interpreting
a single feature from it.

In [ ]:
# === Fetch and inspect the SAE (unvetted community repo — verify before trusting) ===
from huggingface_hub import list_repo_files, hf_hub_download
import torch, json, os

SAE_REPO = "xzascc3944/SAEtopk_Qwen3-4B-Thinking-2507_Layer20"
try:
    files = list_repo_files(SAE_REPO)
    print(f"{SAE_REPO}\nfiles:")
    for f in files:
        print("  ", f)
except Exception as e:
    print("FAILED to list repo:", type(e).__name__, e)
    files = []

# pull config + weights if present
paths = {}
for f in files:
    if f.endswith((".json", ".pt", ".safetensors", ".bin")) and "/" not in f:
        try:
            paths[f] = hf_hub_download(SAE_REPO, f)
            print(f"  downloaded {f} -> {os.path.getsize(paths[f])/2**20:.1f} MiB")
        except Exception as e:
            print(f"  could not fetch {f}: {e}")

for f, p in paths.items():
    if f.endswith(".json"):
        print(f"\n--- {f} ---")
        print(json.dumps(json.load(open(p)), indent=2)[:1200])

# inspect tensor shapes
for f, p in paths.items():
    if f.endswith((".pt", ".bin")):
        sd = torch.load(p, map_location="cpu")
        sd = sd.get("state_dict", sd) if isinstance(sd, dict) else sd
        print(f"\n--- {f} tensors ---")
        for k, v in (sd.items() if isinstance(sd, dict) else []):
            if hasattr(v, "shape"):
                print(f"  {k:40} {tuple(v.shape)}  {v.dtype}")
    elif f.endswith(".safetensors"):
        from safetensors.torch import load_file
        sd = load_file(p)
        print(f"\n--- {f} tensors ---")
        for k, v in sd.items():
            print(f"  {k:40} {tuple(v.shape)}  {v.dtype}")

# === Validate the SAE on OUR activations before drawing any conclusion from it ===
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch, torch.nn.functional as F

print("--- README ---")
try:
    print(open(hf_hub_download(SAE_REPO, "README.md")).read()[:1500])
except Exception as e:
    print("(no README:", e, ")")

sd = load_file(hf_hub_download(SAE_REPO, "SAE_final.safetensors"))
W_enc = sd["encoder.weight"].to(model.device, torch.float32)   # [F, d]
b_enc = sd["encoder.bias"].to(model.device, torch.float32)     # [F]
W_dec = sd["decoder.weight"].to(model.device, torch.float32)   # [d, F]
b_dec = sd["b_dec"].to(model.device, torch.float32)            # [d]
K     = int(sd["k_buffer"].item())
THR   = float(sd["threshold_buffer"].item())
D_SAE = W_enc.shape[0]
print(f"\nd_model={W_enc.shape[1]} d_sae={D_SAE} k={K} threshold={THR:.4f}")

def sae_encode(h):
    """h: [..., d] float32 -> topk sparse codes [..., F]"""
    pre = (h - b_dec) @ W_enc.T + b_enc
    if K > 0:
        v, i = torch.topk(pre, K, dim=-1)
        z = torch.zeros_like(pre).scatter_(-1, i, F.relu(v))
    else:
        z = F.relu(pre - THR)
    return z

def sae_decode(z):
    return z @ W_dec.T + b_dec

# --- collect real activations from OUR prompts, at both candidate hook points ----------
CUES = [" dolphin", " wolf", " panda", " lion", " elephant", " dog", " bear", " fox",
        " horse", " crab", " animal", " tiger", " cat", " eagle"]
@torch.no_grad()
def acts_at(layer_idx, positions="answer"):
    out = []
    for c in CUES:
        o = model(build_ids(_reason_ids(c)), output_hidden_states=True, use_cache=False)
        hs = o.hidden_states[layer_idx][0]
        out.append(hs[-1] if positions == "answer" else hs)
        del o
    return torch.stack(out).float() if positions == "answer" else torch.cat(out).float()

def fve(h):
    z = sae_encode(h); r = sae_decode(z)
    num = ((h - r) ** 2).sum()
    den = ((h - h.mean(0, keepdim=True)) ** 2).sum()
    return 1 - (num / den).item(), (z > 0).float().sum(-1).mean().item()

print(f"\n{'hook':22} {'FVE (answer pos)':>17} {'FVE (all pos)':>14} {'L0':>6}")
print("-"*64)
best = None
for li in [19, 20, 21]:
    h_ans = acts_at(li, "answer")
    h_all = acts_at(li, "all")
    f_ans, l0 = fve(h_ans)
    f_all, _  = fve(h_all)
    print(f"hidden_states[{li}]{'':7} {f_ans:>17.4f} {f_all:>14.4f} {l0:>6.1f}")
    if best is None or f_all > best[1]:
        best = (li, f_all)
SAE_LAYER = best[0]
print(f"\n-> using hidden_states[{SAE_LAYER}] (best all-position FVE = {best[1]:.4f})")
print("   NB: FVE well below ~0.5 would mean this SAE does not describe our activations")
print("   and nothing downstream of it should be believed.")

xzascc3944/SAEtopk_Qwen3-4B-Thinking-2507_Layer20
files:
   .gitattributes
   README.md
   SAE_final.safetensors
   checkpoints/SAE_checkpoint_step_0.safetensors
   checkpoints/SAE_checkpoint_step_1000.safetensors
   checkpoints/SAE_checkpoint_step_2000.safetensors
   checkpoints/SAE_checkpoint_step_3000.safetensors
   checkpoints/SAE_checkpoint_step_4000.safetensors
   checkpoints/SAE_checkpoint_step_5000.safetensors
   checkpoints/SAE_checkpoint_step_6000.safetensors
   checkpoints/SAE_checkpoint_step_7000.safetensors
   checkpoints/SAE_checkpoint_step_8000.safetensors
   checkpoints/SAE_checkpoint_step_9000.safetensors
   checkpoints/SAE_checkpoint_step_9999.safetensors


  downloaded SAE_final.safetensors -> 400.1 MiB

--- SAE_final.safetensors tensors ---
  b_dec                                    (2560,)  torch.float32
  decoder.weight                           (2560, 20480)  torch.float32
  encoder.bias                             (20480,)  torch.float32
  encoder.weight                           (20480, 2560)  torch.float32
  threshold_buffer                         ()  torch.float32
  k_buffer                                 ()  torch.int32


--- README ---


---
license: mit
---


d_model=2560 d_sae=20480 k=230 threshold=0.0044

hook                    FVE (answer pos)  FVE (all pos)     L0
----------------------------------------------------------------
hidden_states[19]          -106587999.0000  -2027698.5000  230.0
hidden_states[20]           -99473967.0000  -2028505.8750  230.0
hidden_states[21]           -95654647.0000  -2027638.1250  230.0

-> using hidden_states[21] (best all-position FVE = -2027638.1250)
   NB: FVE well below ~0.5 would mean this SAE does not describe our activations
   and nothing downstream of it should be believed.


In [ ]:
# === Diagnose: which layer, and what input scaling, does this SAE actually expect? ===
import torch, torch.nn.functional as F, math

@torch.no_grad()
def all_layer_acts(cues=CUES, n_pos=None):
    """Return [L+1, N, d] answer-position activations for every layer."""
    per_layer = None
    for c in cues:
        o = model(build_ids(_reason_ids(c)), output_hidden_states=True, use_cache=False)
        hs = [h[0, -1].float() for h in o.hidden_states]
        if per_layer is None:
            per_layer = [[] for _ in hs]
        for i, h in enumerate(hs):
            per_layer[i].append(h)
        del o
    return torch.stack([torch.stack(v) for v in per_layer])     # [L+1, N, d]

A = all_layer_acts()
print(f"activations {tuple(A.shape)}")
print(f"{'layer':>5} {'mean||h||':>10}")
for li in range(0, A.shape[0], 6):
    print(f"{li:>5} {A[li].norm(dim=-1).mean():>10.2f}")
print(f"{'...':>5}")
print(f"decoder col norm mean: {W_dec.norm(dim=0).mean():.4f}   b_dec norm: {b_dec.norm():.4f}")

def fve_of(h):
    z = sae_encode(h); r = sae_decode(z)
    return 1 - (((h - r)**2).sum() / ((h - h.mean(0, keepdim=True))**2).sum()).item()

SQRT_D = math.sqrt(W_enc.shape[1])
variants = {
    "raw":                     lambda h: h,
    "unit-norm":               lambda h: F.normalize(h, dim=-1),
    "scale to sqrt(d)":        lambda h: h * (SQRT_D / h.norm(dim=-1, keepdim=True).mean()),
    "per-vec sqrt(d)":         lambda h: F.normalize(h, dim=-1) * SQRT_D,
    "std-normalised":          lambda h: (h - h.mean(0, keepdim=True)) / h.std(),
}

print(f"\n{'layer':>5} " + " ".join(f"{k:>17}" for k in variants))
print("-" * (6 + 18*len(variants)))
best = (None, None, -1e18)
for li in range(A.shape[0]):
    row, vals = [], []
    for name, fn in variants.items():
        v = fve_of(fn(A[li]))
        vals.append((name, v)); row.append(f"{v:>17.4f}")
    if li % 4 == 0 or li >= A.shape[0]-2:
        print(f"{li:>5} " + " ".join(row))
    for name, v in vals:
        if v > best[2]:
            best = (li, name, v)
print(f"\nBEST: hidden_states[{best[0]}] with '{best[1]}'  FVE = {best[2]:.4f}")
if best[2] < 0.5:
    print("\n*** No layer/scaling combination reconstructs our activations. This SAE cannot")
    print("*** be used here as-is — do not interpret features from it.")

# === Where exactly does it blow up? (magnitudes, and a few encoder conventions) ===
import torch, torch.nn.functional as F
h = A[20]                                    # [14, 2560]
print(f"||h||          mean {h.norm(dim=-1).mean():.2f}")
print(f"W_enc row norm mean {W_enc.norm(dim=1).mean():.3f}  max {W_enc.norm(dim=1).max():.3f}")
print(f"W_dec col norm mean {W_dec.norm(dim=0).mean():.3f}  (unit-norm decoder: typical)")
print(f"b_enc          mean {b_enc.mean():+.4f}  min {b_enc.min():+.3f}  max {b_enc.max():+.3f}")

def report(name, pre, dec_fn):
    v, i = torch.topk(pre, K, dim=-1)
    z = torch.zeros_like(pre).scatter_(-1, i, F.relu(v))
    r = dec_fn(z)
    print(f"  {name:32} |pre|max={pre.abs().max():>9.2f}  |z|mean={z.sum(-1).mean():>10.2f}  "
          f"||recon||={r.norm(dim=-1).mean():>12.2f}  ratio={r.norm(dim=-1).mean()/h.norm(dim=-1).mean():>8.2f}")

print("\nencoder conventions (layer 20):")
report("(h-b_dec)@Wenc.T + b_enc",  (h - b_dec) @ W_enc.T + b_enc, lambda z: z @ W_dec.T + b_dec)
report("h@Wenc.T + b_enc",          h @ W_enc.T + b_enc,           lambda z: z @ W_dec.T + b_dec)
report("h@Wenc.T (no bias)",        h @ W_enc.T,                   lambda z: z @ W_dec.T + b_dec)
report("unitnorm(h)@Wenc.T + b_enc", F.normalize(h, dim=-1) @ W_enc.T + b_enc, lambda z: z @ W_dec.T + b_dec)

# is the decoder even aligned with the encoder? (tied-ish SAEs have high row/col cosine)
cos = F.cosine_similarity(F.normalize(W_enc, dim=1), F.normalize(W_dec.T, dim=1), dim=1)
print(f"\ncos(enc_row_i, dec_col_i): mean {cos.mean():+.4f}  median {cos.median():+.4f}")
print("  (a trained TopK SAE normally has these strongly positive; ~0 suggests the")
print("   checkpoint is untrained, mismatched, or saved with permuted/!aligned weights)")

# how much of the feature dictionary is even alive?
dead = (W_dec.norm(dim=0) < 1e-6).sum().item()
print(f"dead decoder columns: {dead}/{D_SAE}")

# === Last chance for the SAE: sign/scale conventions × layers ===
import torch, torch.nn.functional as F, math, itertools
SQRT_D = math.sqrt(2560)

def fve_cfg(h, se, sd, norm):
    x = {"raw": h,
         "unit": F.normalize(h, dim=-1),
         "sqrtd": F.normalize(h, dim=-1) * SQRT_D}[norm]
    pre = se * ((x - b_dec) @ W_enc.T + b_enc)
    v, i = torch.topk(pre, K, dim=-1)
    z = torch.zeros_like(pre).scatter_(-1, i, F.relu(v))
    r = sd * (z @ W_dec.T) + b_dec
    return 1 - (((x - r)**2).sum() / ((x - x.mean(0, keepdim=True))**2).sum()).item()

best = (-1e18, None)
for li in [8, 12, 16, 18, 20, 22, 24, 28]:
    for se, sd, norm in itertools.product([1, -1], [1, -1], ["raw", "unit", "sqrtd"]):
        v = fve_cfg(A[li], se, sd, norm)
        if v > best[0]:
            best = (v, (li, se, sd, norm))
print(f"best over {8*2*2*3} configs: FVE={best[0]:.4f}  (layer={best[1][0]}, "
      f"enc_sign={best[1][1]}, dec_sign={best[1][2]}, norm={best[1][3]})")

if best[0] < 0.5:
    print("\n" + "="*72)
    print("VERDICT: this SAE checkpoint is unusable for our activations.")
    print("  - reconstructions ~1000x too large in every standard parameterisation")
    print("  - encoder/decoder pairs anti-aligned (cos = -0.51), which a trained")
    print("    TopK SAE never shows")
    print("  - no layer x sign x normalisation combination reaches even FVE 0.5")
    print("Nothing may be concluded from its features. Not interpreting it further.")
    print("="*72)
else:
    print("usable — proceeding with this configuration")

activations (37, 14, 2560)
layer  mean||h||
    0       0.77
    6      23.21
   12      39.85
   18      46.77
   24      87.61
   30     249.65
   36     176.18
  ...
decoder col norm mean: 1.0000   b_dec norm: 0.1048

layer               raw         unit-norm  scale to sqrt(d)   per-vec sqrt(d)    std-normalised
------------------------------------------------------------------------------------------------
    0 -24159782302377836544.0000 -12099936347674902528.0000 -2547162172010528768.0000 -2563718892979486720.0000              -inf
    4  -8404465663.0000  -1813050239.0000  -8713674751.0000  -8754408447.0000       -78802.3125
    8  -1867182463.0000   -179115951.0000  -1891229183.0000  -1905453567.0000      -133728.0000
   12   -186877695.0000    -13959538.0000   -187271887.0000   -191571567.0000       -95980.7891
   16     -9642040.0000      -861004.0625     -9955658.0000    -10623317.0000      -459976.0312
   20    -99473967.0000     -3356001.0000    -98418623.0000   -108035983

||h||          mean 59.26
W_enc row norm mean 44.060  max 200.985
W_dec col norm mean 1.000  (unit-norm decoder: typical)
b_enc          mean -2.9179  min -7.612  max +5.158

encoder conventions (layer 20):
  (h-b_dec)@Wenc.T + b_enc         |pre|max=   546.58  |z|mean=  64249.71  ||recon||=    58067.39  ratio=  979.80
  h@Wenc.T + b_enc                 |pre|max=   543.94  |z|mean=  64915.52  ||recon||=    58699.30  ratio=  990.46
  h@Wenc.T (no bias)               |pre|max=   537.36  |z|mean=  66213.48  ||recon||=    59894.25  ratio= 1010.62
  unitnorm(h)@Wenc.T + b_enc       |pre|max=    15.14  |z|mean=    194.84  ||recon||=      137.37  ratio=    2.32

cos(enc_row_i, dec_col_i): mean -0.5132  median -0.5244
  (a trained TopK SAE normally has these strongly positive; ~0 suggests the
   checkpoint is untrained, mismatched, or saved with permuted/!aligned weights)
dead decoder columns: 0/20480


best over 96 configs: FVE=-861004.0625  (layer=16, enc_sign=1, dec_sign=1, norm=unit)

VERDICT: this SAE checkpoint is unusable for our activations.
  - reconstructions ~1000x too large in every standard parameterisation
  - encoder/decoder pairs anti-aligned (cos = -0.51), which a trained
    TopK SAE never shows
  - no layer x sign x normalisation combination reaches even FVE 0.5
Nothing may be concluded from its features. Not interpreting it further.


## 7. Does trigger quality track "looks like the real cue"?

The premise behind `grad_lens`, tested directly and without an SAE. For each animal build a
**quality ladder** by reverting *j* of the 8 optimised slots to random pool tokens
(j = 0..8, 3 replicates). That spans p from ~prior to ~max *within* one animal, so the
correlation is not confounded by different ceilings across animals.

Closeness is `cos(h(trigger) − h(neutral), h(real cue) − h(neutral))` at the answer position —
the cue-minus-neutral steering delta, not raw cosine (raw floors near 0.92 because the two
prompts share almost all their context).

Result: mean within-animal r peaks at **+0.86 at layer 32**, and survives residualising both
sides on the degradation level *j* (**partial r = +0.699 pooled**, positive in 8/8 animals).
Junk triggers work by reconstructing the real word's steering direction in the residual
stream. `grad_lens` was aiming at the right quantity — just at layer 18, where the signal is
+0.70 and still climbing, rather than at the layer-32 peak.

**Caveat not tested here:** the neutral reference is always `" animal"`, so every animal's
`DELTA_MID` shares a "generic → specific" component that has nothing to do with the target.
`cos(DELTA_MID[a], DELTA_MID[b])` across targets would put a floor under these numbers.

In [ ]:
# === Your hypothesis, without the SAE: are better triggers closer to the base word? ===
# For each animal build a QUALITY LADDER by reverting j of the 8 optimised slots to random
# pool tokens (j = 0..8, 3 replicates). That spans p from ~prior to ~max within one animal,
# so the correlation is measured WITHIN animal and isn't confounded by different ceilings.
#
# "Closeness to the base word" = cos( h(trigger) - h(neutral) , h(real cue) - h(neutral) )
# at the answer position — the cue-minus-neutral steering delta, not raw cosine (raw floors
# near 0.92 because the two prompts share almost all their context).
import torch, torch.nn.functional as F, math, random

LAYERS = [8, 12, 16, 18, 20, 24, 28, 32, 36]

@torch.no_grad()
def hidden_answer(trigs, chunk=32):
    """[B,k] triggers -> [B, len(LAYERS), d] answer-position hidden states."""
    outs = []
    for i in range(0, trigs.shape[0], chunk):
        blk = trigs[i:i+chunk].to(model.device); B = blk.shape[0]
        ids = torch.cat([PRE.expand(B, -1), blk, SUF.expand(B, -1)], dim=1)
        o = model(ids, output_hidden_states=True, use_cache=False)
        outs.append(torch.stack([o.hidden_states[l][:, -1].float() for l in LAYERS], 1).cpu())
        del o, ids, blk
    return torch.cat(outs)

def pad_cue(txt, k=8):
    """single-token cue padded to k slots so it shares the trigger geometry"""
    ids = tokenizer.encode(txt, add_special_tokens=False)
    return torch.tensor(ids, dtype=torch.long)

rng = random.Random(0)
LADDER = {}
for word in SWEEP_ANIMALS:
    setup_target(word, verbose=False)
    base = SWEEP[word]["runs"]["logit"]["trigger"].clone()
    k = base.numel()
    variants, labels = [], []
    for j in range(k + 1):                       # revert j slots to random pool tokens
        for rep in range(3):
            t = base.clone()
            for s in rng.sample(range(k), j):
                t[s] = POOL[rng.randrange(POOL.numel())]
            variants.append(t); labels.append(j)
    variants = torch.stack(variants)
    p = batch_p_target(variants, 64)
    H = hidden_answer(variants)                                   # [N, L, d]
    h_neu = hidden_answer(pad_cue(" animal")[None])[0]            # [L, d]
    h_real = hidden_answer(pad_cue(" " + word)[None])[0]          # [L, d]
    d_real = h_real - h_neu
    d_trig = H - h_neu[None]
    align = F.cosine_similarity(d_trig, d_real[None], dim=-1)     # [N, L]
    LADDER[word] = dict(p=p, align=align, j=torch.tensor(labels))
    print(f"{word:9} n={len(variants):3d}  p range {p.min():.4f}..{p.max():.4f}")

def corr(a, b):
    a, b = a.double(), b.double()
    a = a - a.mean(); b = b - b.mean()
    return float((a*b).sum() / (a.norm()*b.norm() + 1e-12))

print(f"\nwithin-animal corr( p , alignment-to-base-word ) by layer")
print(f"{'animal':9} " + " ".join(f"L{l:<5}" for l in LAYERS))
print("-"*(10 + 7*len(LAYERS)))
allc = []
for w, d in LADDER.items():
    cs = [corr(d["p"], d["align"][:, i]) for i in range(len(LAYERS))]
    allc.append(cs)
    print(f"{w:9} " + " ".join(f"{c:>+6.2f}" for c in cs))
mean_c = [sum(c[i] for c in allc)/len(allc) for i in range(len(LAYERS))]
print("-"*(10 + 7*len(LAYERS)))
print(f"{'MEAN':9} " + " ".join(f"{c:>+6.2f}" for c in mean_c))
bi = max(range(len(LAYERS)), key=lambda i: mean_c[i])
print(f"\nstrongest at layer {LAYERS[bi]}: mean within-animal r = {mean_c[bi]:+.3f}")

panda     n= 27  p range 0.1106..0.9991
lion      n= 27  p range 0.0422..0.5555
elephant  n= 27  p range 0.0289..0.8059
dog       n= 27  p range 0.0173..0.4797
bear      n= 27  p range 0.0034..0.0812
fox       n= 27  p range 0.0020..0.1431
horse     n= 27  p range 0.0006..0.0081
crab      n= 27  p range 0.0000..0.8249

within-animal corr( p , alignment-to-base-word ) by layer
animal    L8     L12    L16    L18    L20    L24    L28    L32    L36   
-------------------------------------------------------------------------
panda      +0.53  +0.86  +0.95  +0.93  +0.92  +0.98  +0.98  +0.98  +0.95
lion       +0.50  +0.54  +0.30  +0.48  +0.59  +0.63  +0.71  +0.77  +0.32
elephant   +0.56  +0.78  +0.66  +0.66  +0.77  +0.86  +0.86  +0.83  +0.66
dog        +0.50  +0.72  +0.73  +0.70  +0.71  +0.81  +0.85  +0.83  +0.46
bear       +0.30  +0.18  +0.46  +0.68  +0.83  +0.84  +0.85  +0.82  -0.32
fox        +0.39  -0.01  +0.19  +0.55  +0.58  +0.85  +0.85  +0.85  +0.19
horse      +0.81  +0.91  +0.90  +0.9

In [ ]:
# === Control: does alignment predict p BEYOND the degradation level j? ===
# Both p and alignment fall as we revert more slots, so the raw correlation is partly a
# shared trend. Residualise both on j (subtract within-j means) and re-correlate.
import torch

def corr(a, b):
    a, b = a.double(), b.double(); a = a - a.mean(); b = b - b.mean()
    return float((a*b).sum() / (a.norm()*b.norm() + 1e-12))

def residualise(x, j):
    r = x.clone().double()
    for lvl in j.unique():
        m = (j == lvl)
        r[m] = r[m] - r[m].mean()
    return r

print(f"{'animal':9} {'raw r (L32)':>12} {'partial r | j':>14} {'r(p,j)':>9} {'r(align,j)':>11}")
print("-"*60)
L32 = LAYERS.index(32)
raws, parts = [], []
for w, d in LADDER.items():
    p, al, j = d["p"].double(), d["align"][:, L32].double(), d["j"]
    raw = corr(p, al)
    part = corr(residualise(p, j), residualise(al, j))
    raws.append(raw); parts.append(part)
    print(f"{w:9} {raw:>+12.3f} {part:>+14.3f} {corr(p, j.double()):>+9.3f} {corr(al, j.double()):>+11.3f}")
print("-"*60)
print(f"{'MEAN':9} {sum(raws)/len(raws):>+12.3f} {sum(parts)/len(parts):>+14.3f}")

# pooled across all animals, z-scored within animal so ceilings don't dominate
allp, alla = [], []
for w, d in LADDER.items():
    p, al, j = d["p"].double(), d["align"][:, L32].double(), d["j"]
    rp, ra = residualise(p, j), residualise(al, j)
    allp.append((rp - rp.mean())/(rp.std()+1e-12)); alla.append((ra - ra.mean())/(ra.std()+1e-12))
allp, alla = torch.cat(allp), torch.cat(alla)
print(f"\npooled partial correlation (n={allp.numel()}): r = {corr(allp, alla):+.3f}")
print(f"positive partial r in {sum(x>0 for x in parts)}/{len(parts)} animals")

animal     raw r (L32)  partial r | j    r(p,j)  r(align,j)
------------------------------------------------------------
panda           +0.982         +0.932    -0.800      -0.815
lion            +0.771         +0.450    -0.713      -0.791
elephant        +0.828         +0.495    -0.857      -0.800
dog             +0.829         +0.428    -0.842      -0.928
bear            +0.824         +0.873    -0.692      -0.460
fox             +0.854         +0.763    -0.790      -0.766
horse           +0.901         +0.831    -0.836      -0.738
crab            +0.881         +0.821    -0.738      -0.828
------------------------------------------------------------
MEAN            +0.859         +0.699

pooled partial correlation (n=216): r = +0.699
positive partial r in 8/8 animals
